# 12 — Compare LCIA scores across databases

**Audience:** Users checking how LCIA scores differ between a source ecoinvent database and one or more Premise databases.

**Prerequisites:** The databases and LCIA methods must already be registered in one Brightway project.

**Learning goals:** select comparable methods, run `comparative_analysis` on named databases, save a compact result, and reuse the function with a `NewDatabase` instance.


## Outline

1. Validate project databases and methods.
2. Compare a bounded number of common datasets.
3. Export the result and define an in-memory variant.


In [ ]:
from pathlib import Path

import bw2data as bd

from premise.score_comparison import comparative_analysis

PROJECT = "ecoinvent-3.12-cutoff"
DATABASES = [
    "ecoinvent-3.12-cutoff",
    "premise-remind-ssp2-ndc-2030",
]
OUTPUT = Path("export/score-comparison/datasets_comparison.xlsx")

bd.projects.set_current(PROJECT)
missing = [name for name in DATABASES if name not in bd.databases]
if missing:
    raise ValueError(f"Missing Brightway databases: {missing}")


## 1. Select indicators

Start with one or two methods while validating the workflow. Exact tuples are preferable in reproducible studies.


In [ ]:
INDICATORS = [
    method
    for method in bd.methods
    if "ef v3.1" in str(method).lower() and "climate change" in str(method).lower()
][:2]
if not INDICATORS:
    raise ValueError("No matching EF v3.1 climate-change methods found.")
INDICATORS


## 2. Compare named databases

`limit` bounds the number of common datasets and is useful for a first smoke test. Set it to `None` only when you intend to process the full intersection.


In [ ]:
comparison = comparative_analysis(
    databases=DATABASES,
    indicators=INDICATORS,
    limit=100,
)
comparison.head()


In [ ]:
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
comparison.to_excel(OUTPUT, index=False)
OUTPUT


## 3. Variant for an in-memory build

If the current session still has a `NewDatabase` instance, pass it directly instead of naming databases. The helper below makes that dependency explicit rather than relying on hidden notebook state.


In [ ]:
def compare_new_database(ndb, indicators, limit=100):
    return comparative_analysis(
        ndb=ndb,
        indicators=indicators,
        limit=limit,
    )


## Pitfalls and extension

- Database names and method tuples must match the active project exactly.
- A low `limit` is a smoke test, not a representative sample unless selection is designed explicitly.
- Large comparisons can be slow because each common dataset is characterized repeatedly.

## Exercise

Select one additional non-climate indicator and compare only direct emissions using the `direct_only` argument.


In [ ]:
exercise_direct_only = True
exercise_limit = 25
